## Análise inicial 

In [0]:
%sql
-- Analise das colunas da tabela e seus tipos 
DESCRIBE lh_nautical.lh_nautical_db.clientes_crm;

-- Amostra de dados para análise
SELECT * FROM lh_nautical.lh_nautical_db.clientes_crm;

code,email,full_name,location
1,femininos.oliveira.antunes@icloud.com,Femininos Oliveira Antunes,"Aratu (Candeias) , BA"
2,nunes.fernanda.soares.azevedo.vieira@outlook.com,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife"
3,farias.teixeira.daniel.ribeiro#gmail.com,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS"
4,thiago.moreira#gmail.com,Thiago Moreira,"AC , Rio Branco"
5,pedro.freitas#icloud.com,Pedro Freitas,PA - Santarém Novo
6,coelho.pinheiro.peixoto.antônia.cavalcanti@aol.com,Antônia Coelho Pinheiro Peixoto Cavalcanti,"Fortaleza do Tabocão , TO"
7,torres.barros.rocha.bianca.siqueira#aol.com,Bianca Barros Rocha Torres Siqueira,PB/Cabedelo
8,pimentel.alves.luiz#outlook.com,Luiz Alves Pimentel,SE - Aracaju
9,lucas.lopes.guedes.cunha#tutanota.com,Lucas Guedes Cunha Lopes,PB - João Pessoa
10,paiva.débora#gmx.com,Débora Paiva,Santarém / PA


## Limpar e estruturar a tabela de clientes

In [0]:
%sql
-- Limpar e estruturar a tabela de Clientes (Camada Prata).
-- Regras aplicadas:
-- Deduplicação e remoção de IDs nulos.
-- Email: Substituição de '#' por '@' e padronização para minúsculo (LOWER).
-- Nome: Remoção de espaços vazios nas pontas (TRIM).
-- Localização: Extração da UF e limpeza do restante para a cidade e criar 
-- campo de referencia.
-- ==============================================================================

CREATE OR REPLACE TABLE lh_nautical.lh_nautical_db.slv_clientes AS
WITH cte_clientes_deduplicados AS (
    -- Filtro de Nulos e Duplicatas
    SELECT DISTINCT * FROM lh_nautical.lh_nautical_db.clientes_crm
    WHERE code IS NOT NULL
)
-- Transformação e Estruturação
SELECT 
    code AS id_client,
    TRIM(full_name) AS client_name,
    
    -- Limpeza do E-mail: Troca o # e força tudo para minúsculo
    LOWER(REPLACE(email, '#', '@')) AS email_clean,
    
    -- Extração do Estado (UF): Pega apenas as 2 letras maiúsculas
    REGEXP_EXTRACT(location, '([A-Z]{2})', 1) AS state,
    
    -- Extração do Porto
    CASE 
          WHEN location LIKE '%(%' THEN 
              TRIM(
                  REGEXP_REPLACE( -- Apaga pontuações
                      REGEXP_REPLACE(REGEXP_EXTRACT(location, '^([^\\(]+)', 1), '([A-Z]{2})', ''), -- Apaga a UF
                  '[,\\-/\\\\]', '')
              )
          ELSE NULL 
      END AS port,
    
    
   -- Limpeza da cidade - virgula,traços,barras e remove a referencia de porto
   -- que algumas cidades tem
   CASE 
        WHEN location LIKE '%(%' THEN TRIM(REGEXP_EXTRACT(location, '\\(([^)]+)\\)', 1))
        ELSE TRIM(REGEXP_REPLACE(REGEXP_REPLACE(location, '([A-Z]{2})', ''), '[,\\-/\\\\]', ''))
    END AS city
    
FROM cte_clientes_deduplicados;

-- Etapa 3: Validação
SELECT * FROM lh_nautical.lh_nautical_db.slv_clientes LIMIT 10;

id_client,client_name,email_clean,state,port,city
35,Priscila Martins,martins.priscila@tutanota.com,PB,null,João Pessoa
38,Mateus Antunes,mateus.antunes@gmail.com,TO,null,Fortaleza do Tabocão
27,Flávia Nunes Martins Ribeiro Coelho,nunes.flávia.coelho.martins.ribeiro@protonmail.com,BA,null,Salvador
39,Renata Oliveira Siqueira,oliveira.renata.siqueira@aol.com,PE,Suape,Ipojuca
21,Sandra Rocha Oliveira Soares Peixoto,peixoto.soares.rocha.oliveira.sandra@yahoo.com,AC,null,Rio Branco
32,Daniela Teixeira Antunes Coelho Batista,antunes.coelho.batista.teixeira.daniela@hotmail.com,BA,null,Porto Seguro
41,Pedro Ribeiro Costa Santos Correia,costa.correia.ribeiro.santos.pedro@yahoo.com,BA,Aratu,Candeias
46,Ana Silva Costa Farias Coelho,costa.farias.coelho.ana.silva@hotmail.com,AP,null,Macapá
22,Daniela Borges Vieira Farias Mendonça,borges.vieira.daniela.mendonça.farias@protonmail.com,SE,null,Aracaju
43,Jéssica Farias Cunha,farias.jéssica.cunha@outlook.com,MA,Itaqui,São Luís
